# Вибір ML-моделі для класифікації Hill–Valley

**Мета ноутбука:** провести повний цикл інженерного вибору моделі — від формулювання
вимог і кандидатів до фінальної перевірки на test set — за 16 розділами завдання.

**Що важливо запам'ятати перед запуском:**
- Усі preprocessing-кроки живуть **всередині `Pipeline`** — інакше буде data leakage.
- Test set відділяється **один раз** і «запечатується» до розділу 14.
- Кандидати, ваги та межі **не змінюються** після перегляду test-результатів.
- Усі вимірювання latency виконуються в **однакових умовах** (той самий batch, той самий CPU/GPU).

**Датасет:** `Hill_Valley_without_noise_Training.data` — бінарна класифікація форми сигналу.

**Що ми зробили:**
1. Завантажили датасет, розділили на `X` (усі стовпці, крім останнього) та `y` (останній стовпець).
2. Зафіксували `RANDOM_STATE=42` — це гарантує, що всі наступні запуски дадуть **той самий** результат.
3. Зробили стратифіковане розділення 80/20: `stratify=y` зберігає пропорцію класів у train/test.

**Чому test set відділяється одразу:**
- Інакше будь-який крок (наприклад, підбір гіперпараметрів на CV) міг би «підглянути» в test-дані.
- Після розділення `X_test`/`y_test` **не торкаємось** до клітинки 14.

**На що звернути увагу у виводі:**
- Якщо `X.shape[1]` велике (≈100) — це очікувано: датасет Hill-Valley має ~100 точок на криву.
- Розподіл класів має бути близький до 50/50 — якщо ні, варто подумати про `class_weight`.

In [50]:
# Cell 1: Imports and data loading
from pathlib import Path
import json
import pickle
import time
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import json
import pickle
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


warnings.filterwarnings("ignore")

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42

# Load data (letting pandas automatically handle the header rows)
tr = pd.read_csv("Hill_Valley_without_noise_Training.data")
te = pd.read_csv("Hill_Valley_without_noise_Testing.data")
all_df = pd.concat([tr, te], ignore_index=True)

# Hill-Valley class labels are in the last column
# The values are already numeric 0 and 1 (or 1 and 2), so we convert them cleanly
y = all_df.iloc[:, -1].astype(int)
X = all_df.iloc[:, :-1]

# Perform the clean stratified split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train_full.shape, "| Test shape:", X_test.shape)
print("Class distribution in Train:\n", y_train_full.value_counts())


Train shape: (969, 100) | Test shape: (243, 100)
Class distribution in Train:
 class
1    489
0    480
Name: count, dtype: int64


## Розділ 1 — Інженерний контекст

| Пункт | Опис |
|---|---|
| **Задача** | Бінарна класифікація форми сигналу (пагорб vs долина). |
| **Target** | `y ∈ {0, 1}`. |
| **Позитивний клас** | `1` — клас, для якого критичний **recall** (пропуск = дорога помилка). |
| **Споживач прогнозу** | REST-сервіс реального часу; клієнт — внутрішній дашборд. |
| **Середовище** | CPU-only, single-process inference, batch = 200 зразків. |
| **Режим** | Синхронний (онлайн-скоринг). |
| **Ціна критичної помилки** | Високий FN-cost → пріоритет ROC-AUC і recall. |
| **Обмеження** | p95-latency ≤ 50 мс, train ≤ 60 с, артефакт ≤ 100 МБ. |
| **Умови перевибору** | PSI > 0.2; падіння ROC-AUC на моніторі > 5 п.п.; оновлення SLA; ретрейн > 1×/міс. |

**Чому вимоги вимірювані:**
Формулювання «модель має бути швидкою» перетворюється на конкретну метрику:
`p95_predict_ms_max = 50`. Без числа не можна перевірити обмеження програмно.

In [51]:
CANDIDATES = {
    "dummy": {
        "pipeline": Pipeline([("clf", DummyClassifier(strategy="prior",
                                                      random_state=RANDOM_STATE))]),
        "preprocess": "—",
        "algorithm": "DummyClassifier",
        "libs": {"scikit-learn": ">=1.3"},
    },
    "logreg": {
        "pipeline": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, C=1.0,
                                       random_state=RANDOM_STATE)),
        ]),
        "preprocess": "median-impute + StandardScaler",
        "algorithm": "LogisticRegression(C=1.0)",
        "libs": {"scikit-learn": ">=1.3"},
    },
    "knn": {
        "pipeline": Pipeline([
            ("sc", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5)),
        ]),
        "preprocess": "StandardScaler",
        "algorithm": "KNN(k=5)",
        "libs": {"scikit-learn": ">=1.3"},
    },
    "svc_rbf": {
        "pipeline": Pipeline([
            ("sc", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=2.0, gamma="scale",
                        probability=True, random_state=RANDOM_STATE)),
        ]),
        "preprocess": "StandardScaler",
        "algorithm": "SVC(rbf, C=2, probability=True)",
        "libs": {"scikit-learn": ">=1.3"},
    },
    "rf": {
        "pipeline": Pipeline([
            ("clf", RandomForestClassifier(
                n_estimators=300, max_depth=None, n_jobs=-1,
                random_state=RANDOM_STATE)),
        ]),
        "preprocess": "—",
        "algorithm": "RandomForest(300)",
        "libs": {"scikit-learn": ">=1.3"},
    },
}
print("Кандидатів:", list(CANDIDATES))

Кандидатів: ['dummy', 'logreg', 'knn', 'svc_rbf', 'rf']


**Що зафіксовано для кожного кандидата:**
- `preprocess` — кроки обробки (imputation, scaling);
- `algorithm` — клас алгоритму з ключовими гіперпараметрами;
- `random_state` — для відтворюваності;
- поріг рішення — тут скрізь стандартний (`0.5`), бо класи збалансовані;
- версії бібліотек — фіксуються в описі, для повної відтворюваності у продакшені
  потрібен `requirements.txt`.

**Чому саме такі кандидати:**
| Модель | Роль у наборі |
|---|---|
| `dummy` | Нижня межа (baseline). Показує, чи модель взагалі щось навчилась. |
| `logreg` | Лінійна, швидка, інтерпретована — типовий «безпечний» вибір для продакшену. |
| `knn` | Непараметрична, чутлива до scaling — перевіряє гіпотезу локальної структури. |
| `svc_rbf` | Нелінійна, потужна на середніх даних, але дорога при inference. |
| `rf` | Ансамбль, стійкий до шуму, але важчий за розміром. |

**Правило:** після відкриття test set **жоден** з цих кандидатів не змінюється.
Додати нового кандидата можна лише як нову версію експерименту.

In [ ]:
from sklearn.base import clone 

def evaluate_candidate(pipe, X, y, cv):
    fold_metrics = []
    for tr, te in cv.split(X, y):
        m = clone(pipe)
        m.fit(X.iloc[tr], y.iloc[tr])
        yp = m.predict(X.iloc[te])
        yp_proba = m.predict_proba(X.iloc[te])[:, 1]
        fold_metrics.append({
            "accuracy": accuracy_score(y.iloc[te], yp),
            "f1": f1_score(y.iloc[te], yp),
            "roc_auc": roc_auc_score(y.iloc[te], yp_proba),
        })
    df = pd.DataFrame(fold_metrics)
    agg = {}
    for c in df.columns:
        agg[f"{c}_mean"] = float(df[c].mean())
        agg[f"{c}_std"] = float(df[c].std(ddof=0))
    return agg, fold_metrics

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, spec in CANDIDATES.items():
    agg, folds = evaluate_candidate(spec["pipeline"], X_train_full, y_train_full, CV)
    cv_results[name] = {"agg": agg, "folds": folds}
    print(f"{name:10s}  ROC-AUC={agg['roc_auc_mean']:.3f}±{agg['roc_auc_std']:.3f}"
          f"  Acc={agg['accuracy_mean']:.3f}  F1={agg['f1_mean']:.3f}")

dummy       ROC-AUC=0.500±0.000  Acc=0.505  F1=0.671
logreg      ROC-AUC=0.959±0.023  Acc=0.681  F1=0.758
knn         ROC-AUC=0.566±0.028  Acc=0.558  F1=0.570
svc_rbf     ROC-AUC=0.479±0.069  Acc=0.514  F1=0.644


**Ключові інженерні рішення:**
- Один `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` — **той самий** для всіх
  кандидатів. Це критично: інакше різні моделі «змагались» би на різних фолдах і порівняння
  було б некоректним.
- `clone(pipe)` всередині циклу — не перевикористовуємо fitted стан між фолдами.
- Preprocessing живе в pipeline → **немає data leakage**: `StandardScaler` і `SimpleImputer`
  навчаються лише на train-частині фолда.

**Як читати вивід:**
| Стовпець | Що означає |
|---|---|
| `ROC-AUC mean ± std` | Основна метрика. Std показує стабільність між фолдами. |
| `Accuracy mean` | Порогова метрика; при збалансованих класах узгоджується з ROC-AUC. |
| `F1 mean` | Компроміс precision/recall. |

**Інженерний критерій:** std > 0.03 за ROC-AUC — модель нестабільна, потрібно більше фолдів
або регуляризація.

**Типові очікування для Hill-Valley:** `logreg` дасть ~0.9+, `svc_rbf` і `rf` — близько 0.95+;
`dummy` — 0.5 (випадковий).

In [ ]:
def profile_candidate(pipe, X_train, y_train, X_batch,
                      n_warmup=5, n_calls=30):
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time_s = time.perf_counter() - t0

    for _ in range(n_warmup):
        pipe.predict(X_batch)

    times = []
    for _ in range(n_calls):
        t0 = time.perf_counter()
        pipe.predict(X_batch)
        times.append((time.perf_counter() - t0) * 1000.0)
    times = np.asarray(times)

    size_mb = len(pickle.dumps(pipe)) / 1e6
    throughput = len(X_batch) / (times.mean() / 1000.0)
    return {
        "train_time_s": train_time_s,
        "p50_predict_ms": float(np.percentile(times, 50)),
        "p95_predict_ms": float(np.percentile(times, 95)),
        "throughput_sps": float(throughput),
        "artifact_size_mb": float(size_mb),
        "times_ms": times,
    }

BATCH_SIZE = 200
rng = np.random.RandomState(RANDOM_STATE)
batch_idx = rng.choice(len(X_train_full), size=BATCH_SIZE, replace=False)
X_batch = X_train_full.iloc[batch_idx]
print(f"Батч={BATCH_SIZE} зразків | profiling env")

profiling = {}
for name, spec in CANDIDATES.items():
    p = profile_candidate(clone(spec["pipeline"]), X_train_full, y_train_full, X_batch)
    profiling[name] = p
    print(f"{name:10s}  train={p['train_time_s']:6.2f}s  "
          f"p50={p['p50_predict_ms']:6.2f}ms  p95={p['p95_predict_ms']:7.2f}ms  "
          f"size={p['artifact_size_mb']:7.3f}MB  thr={p['throughput_sps']:9.0f} sps")

**Чому саме так міряємо:**
| Крок | Обґрунтування |
|---|---|
| 5 прогрівів | Перші виклики повільніші через JIT, lazy-import, ініціалізацію бібліотек. Їх не враховуємо. |
| 30 вимірюваних викликів | Достатньо для стабільної оцінки p95 у `numpy.percentile`. |
| Один і той самий `X_batch` | Порівняння справедливе лише за однакових вхідних даних. |
| `perf_counter()` | Монотонний таймер з високою точністю. |
| Фіксований batch = 200 | Узгоджуємо з SLA сервісу. |

**Що показує кожна метрика:**
- `train_time_s` — скільки часу чекати CI/CD-пайплайн при ретрейні;
- `p50_predict_ms` — медіана, «звичайний» досвід користувача;
- `p95_predict_ms` — «хвіст», визначає SLA і розмір черги;
- `throughput_sps` — скільки зразків за секунду обробить сервіс;
- `artifact_size_mb` — впливає на час завантаження моделі, розмір Docker-образу.

**Обмеження методу:**
- Пам'ять **не вимірюється** — фіксуємо лише розмір артефакту. Для RSS/peak RSS
  потрібен окремий інструмент (`memory_profiler`, `tracemalloc`).
- Заміри залежать від навантаження на CPU; для стабільності — виконувати на «тихій» машині.
- `svc_rbf` з `probability=True` має дорожчий `predict_proba` — профілюємо саме його,
  бо саме `predict_proba` використовується для ROC-AUC у CV.

In [ ]:
# Ensure the artifacts directory is defined and exists
OUT = Path("artifacts")
OUT.mkdir(exist_ok=True)

rows = []
for name in CANDIDATES:
    row = {"model": name}
    row.update(cv_results[name]["agg"])
    p = profiling[name]
    row.update({k: v for k, v in p.items() if k != "times_ms"})
    rows.append(row)
benchmark_df = pd.DataFrame(rows).round(6)
benchmark_df.to_csv(OUT / "benchmark_results.csv", index=False)
print("Збережено:", OUT / "benchmark_results.csv")
benchmark_df

**Структура:** один рядок = один кандидат. Стовпці:
- `model` — унікальна назва (гарантовано, бо приходить з ключів словника);
- `*_mean`, `*_std` — агрегати CV;
- `train_time_s`, `p50_predict_ms`, `p95_predict_ms`, `throughput_sps`, `artifact_size_mb`.

**Перевірка на «непояснені пропуски»:** після збору переконайтеся, що `df.isna().sum()` = 0.
Якщо десь NaN — це сигнал, що якась метрика не порахувалася (наприклад, `predict_proba`
відсутній у кандидата).

**Швидка інтерпретація таблиці:**
- Перетин `roc_auc_mean` (високе) × `p95_predict_ms` (низьке) — «золота зона»;
- `svc_rbf` зазвичай має найкращий ROC-AUC, але й найгірший p95;
- `logreg` — компроміс: AUC близько 0.9, latency ~1–3 мс;
- `rf` — AUC висока, але `artifact_size_mb` десятки мегабайт.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(benchmark_df["p95_predict_ms"], benchmark_df["roc_auc_mean"],
           s=80, c="tab:blue")
for _, r in benchmark_df.iterrows():
    ax.annotate(r["model"], (r["p95_predict_ms"], r["roc_auc_mean"]),
                xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("p95 часу прогнозування, мс")
ax.set_ylabel("CV ROC-AUC (mean)")
ax.set_title("Якість проти затримки")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "quality_vs_latency.png", dpi=120); plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for name in CANDIDATES:
    ax.boxplot(profiling[name]["times_ms"],
               positions=[list(CANDIDATES).index(name)],
               widths=0.5, showfliers=False)
ax.set_xticks(range(len(CANDIDATES))); ax.set_xticklabels(CANDIDATES)
ax.set_ylabel("Час прогнозування батча, мс")
ax.set_title("Розподіл 30 повторних вимірювань (без прогріву)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "latency_distribution.png", dpi=120); plt.show()

**Графік 1 — якість проти p95-latency:**
- Лівий верхній кут — мрія: висока якість + низька затримка.
- Пунктирні орієнтири SLA (50 мс) і мінімального ROC-AUC (0.85) відсікають «допустиму зону».
- Моделі праворуч від SLA-межі — кандидати на відхилення.

**Графік 2 — розподіл 30 вимірювань (box-plot):**
- Вузький ящик = стабільна затримка → передбачуваний SLA.
- Широкий ящик або «вусики» = висока дисперсія → ризик випадкових піків.
- Викиди приховані (`showfliers=False`), бо мета — побачити форму розподілу.

**Інженерний критерій:** якщо p95/p50 > 2 — розподіл «товстохвостий», тобто SLA
потрібно проєктувати з запасом навіть при хорошій медіані.

In [ ]:
SCENARIO_FAST = {
    "scenario_name": "fast_inference",
    "config_version": "1.0",
    "description": "Пріоритет швидкодії інференсу, мінімально прийнятна якість",
    "constraints": {
        "accuracy_min": 0.80,
        "roc_auc_min": 0.85,
        "train_time_s_max": 60.0,
        "p95_predict_ms_max": 50.0,
        "artifact_size_mb_max": 100.0,
    },
    "criteria": {
        "roc_auc":          {"direction": "maximize", "weight": 0.40, "min_ref": 0.50, "max_ref": 1.00},
        "p95_predict_ms":   {"direction": "minimize", "weight": 0.40, "min_ref": 0.50, "max_ref": 200.0},
        "artifact_size_mb": {"direction": "minimize", "weight": 0.20, "min_ref": 0.01, "max_ref": 100.0},
    },
    "refs_source": "min_ref — baseline dummy (ROC-AUC 0.5), max_ref — теоретичний максимум; "
                   "ліміти latency/size — вимоги SLA сервісу",
}

SCENARIO_QUALITY = {
    "scenario_name": "quality_first",
    "config_version": "1.0",
    "description": "Пріоритет якості та інтерпретованості",
    "constraints": {
        "accuracy_min": 0.85,
        "roc_auc_min": 0.90,
        "p95_predict_ms_max": 200.0,
        "artifact_size_mb_max": 200.0,
    },
    "criteria": {
        "roc_auc":        {"direction": "maximize", "weight": 0.60, "min_ref": 0.50, "max_ref": 1.00},
        "accuracy":       {"direction": "maximize", "weight": 0.30, "min_ref": 0.50, "max_ref": 1.00},
        "p95_predict_ms": {"direction": "minimize", "weight": 0.10, "min_ref": 0.50, "max_ref": 200.0},
    },
    "refs_source": "min_ref/max_ref успадковано з попереднього експерименту та SLA",
}

def validate_config(cfg):
    assert "criteria" in cfg and len(cfg["criteria"]) > 0, "criteria порожні"
    w = [c["weight"] for c in cfg["criteria"].values()]
    assert abs(sum(w) - 1.0) < 1e-6, f"Сума ваг = {sum(w):.6f}, має бути 1.0"
    for name, c in cfg["criteria"].items():
        assert c["direction"] in ("maximize", "minimize"), f"{name}: погане direction"
        assert c["min_ref"] < c["max_ref"], f"{name}: min_ref >= max_ref"

validate_config(SCENARIO_FAST)
validate_config(SCENARIO_QUALITY)
(OUT / "scenario_fast.json").write_text(json.dumps(SCENARIO_FAST, ensure_ascii=False, indent=2))
(OUT / "scenario_quality.json").write_text(json.dumps(SCENARIO_QUALITY, ensure_ascii=False, indent=2))
print("Конфігурації валідовані та збережені")

**Ключова ідея:** важка логіка (ваги, обмеження, орієнтири) виноситься в JSON/YAML —
Python-код не змінюється. Це дозволяє **окреслити сценарії без релізу коду**.

**Два сценарії:**
| Аспект | `fast_inference` | `quality_first` |
|---|---|---|
| Головний критерій | p95-latency (вага 0.4) | ROC-AUC (вага 0.6) |
| Ліміт p95 | 50 мс | 200 мс |
| Ліміт розміру | 100 МБ | 200 МБ |
| Мінімальний ROC-AUC | 0.85 | 0.90 |

**Що таке `min_ref` / `max_ref` і звідки вони:**
- Це межі нормалізації. Значення поза ними «обрізаються» до 0 або 1.
- `roc_auc`: min=0.5 (випадковий baseline), max=1.0 (теоретичний максимум).
- `p95_predict_ms`: min=0.5 мс (швидкість dummy/logreg), max=200 мс (межа SLA).
- Джерело: **SLA сервісу + попередній досвід**; зафіксовано в `refs_source`.

**Чому `direction` у конфігу, а не виводиться з назви метрики:**
- Назва метрики не дає однозначної семантики. Наприклад, `error` = «менше краще»,
  але `auc` = «більше краще». Явне поле `direction` усуває двозначність і дозволяє
  програмно перевірити конфіг.

**Перевірки при завантаженні:**
- Сума ваг = 1.0;
- Кожен критерій має `min_ref < max_ref`;
- `direction ∈ {maximize, minimize}`.

In [ ]:
def check_constraints(metrics: dict, constraints: dict) -> tuple[bool, list[str]]:
    viol = []
    for key, limit in constraints.items():
        if key.endswith("_max"):
            m = key[:-4]           # "accuracy_max" -> "accuracy"
            if m not in metrics or pd.isna(metrics[m]):
                viol.append(f"Відсутня метрика: {m}"); continue
            if metrics[m] > limit:
                viol.append(f"{m}={metrics[m]:.4f} > {limit}")
        elif key.endswith("_min"):
            m = key[:-4]           # ✅ БУЛО -3, СТАЛО -4
            if m not in metrics or pd.isna(metrics[m]):
                viol.append(f"Відсутня метрика: {m}"); continue
            if metrics[m] < limit:
                viol.append(f"{m}={metrics[m]:.4f} < {limit}")
        else:
            viol.append(f"Невідомий формат обмеження: {key}")
    return len(viol) == 0, viol


def normalize(value: float, direction: str, min_ref: float, max_ref: float) -> float:
    if pd.isna(value):
        return 0.0
    if direction == "maximize":
        v = (value - min_ref) / (max_ref - min_ref)
    elif direction == "minimize":
        v = (max_ref - value) / (max_ref - min_ref)
    else:
        raise ValueError(f"Невідомий напрям: {direction}")
    return float(np.clip(v, 0.0, 1.0))

**`check_constraints` — правила:**
1. **Значення точно на межі — допустиме.** `roc_auc = 0.85` при `roc_auc_min = 0.85` → OK.
   Це узгоджується з інженерним «≥ / ≤» у SLA.
2. **Повертаються ВСІ порушення**, а не перше. Це важливо для діагностики: якщо модель
   провалила одночасно latency і розмір — інженер має побачити обидві причини.
3. **Відсутня метрика = порушення.** Не можна мовчки пропустити критерій.
4. **Суфікси `_min` / `_max`** розділяють напрямок перевірки, а не напрямок критерію.

**`normalize` — правила:**
- `maximize`: `(value - min_ref) / (max_ref - min_ref)`;
- `minimize`: `(max_ref - value) / (max_ref - min_ref)`;
- **clip до [0, 1]** — значення за межами орієнтирів не ламають шкалу;
- напрямок передається **явно** через параметр, не виводиться з назви метрики.

**Приклади:**
| value | direction | min_ref | max_ref | normalize |
|---|---|---|---|---|
| 0.95 | maximize | 0.5 | 1.0 | 0.9 |
| 10 | minimize | 0.5 | 200 | ~0.95 |
| 0.4 | maximize | 0.5 | 1.0 | 0.0 (clip) |
| 500 | minimize | 0.5 | 200 | 0.0 (clip) |

In [ ]:
def rank_models(results: pd.DataFrame, config: dict):
    validate_config(config)
    constraints = config["constraints"]
    criteria = config["criteria"]

    valid, rejected = [], []
    for _, row in results.iterrows():
        m = row.to_dict()
        ok, viol = check_constraints(m, constraints)
        if not ok:
            rejected.append({"model": m["model"], "violations": "; ".join(viol)})
            continue

        missing = [c for c in criteria if c not in m or pd.isna(m[c])]
        if missing:
            rejected.append({"model": m["model"],
                             "violations": f"Відсутні метрики критеріїв: {missing}"})
            continue

        entry = {"model": m["model"]}
        total = 0.0
        for cname, spec in criteria.items():
            s = normalize(m[cname], spec["direction"], spec["min_ref"], spec["max_ref"])
            entry[f"norm_{cname}"] = round(s, 6)
            total += spec["weight"] * s
        entry["score"] = round(total, 6)
        valid.append(entry)

    if len(valid) == 0:
        return (pd.DataFrame(columns=["model", "score", "rank"]),
                pd.DataFrame(rejected))

    valid_df = (pd.DataFrame(valid)
                .sort_values(["score", "model"], ascending=[False, True])
                .reset_index(drop=True))
    valid_df["rank"] = valid_df.index + 1
    rejected_df = pd.DataFrame(rejected) if rejected else pd.DataFrame(
        columns=["model", "violations"])
    return valid_df, rejected_df

**Алгоритм `rank_models`:**
1. Пройти по кожному кандидату.
2. Якщо порушує хоч одне обмеження → у `rejected` (з переліком усіх причин).
3. Якщо відсутня метрика критерію → теж у `rejected` (модель не можна оцінити).
4. Інакше — нормалізувати кожен критерій, помножити на вагу, додати в `score`.
5. Відсортувати за `score` спадно, при рівності — за `model` (детермінізм).

**Чому сортування другою ознакою — назва моделі:**
- Однакові бали — часто реальність (дві гарні моделі).
- Сортування за алфавітом робить результат **відтворюваним** між запусками.
- Альтернатива — за `p95_predict_ms`, але тоді треба явно зафіксувати policy в коді.

**Дві таблиці, два підсумки:**
- `valid_df`: `model, norm_<crit>..., score, rank` — **тільки** допустимі моделі.
- `rejected_df`: `model, violations` — усі причини відхилення, об'єднані в рядок.

**Що робити, якщо `valid_df` порожній:**
- Це валідний результат. Сценарій вважається нерозв'язним з наявними кандидатами.
- Потрібно або послабити обмеження, або додати нових кандидатів (як **нову** версію
  експерименту, а не «на льоту»).

In [ ]:
def persist_scenario(scenario_cfg, benchmark_df, tag):
    valid_df, rejected_df = rank_models(benchmark_df, scenario_cfg)
    valid_df.to_csv(OUT / f"ranking_{tag}.csv", index=False)
    rejected_df.to_csv(OUT / f"rejected_{tag}.csv", index=False)

    summary = {
        "scenario_name": scenario_cfg["scenario_name"],
        "config_version": scenario_cfg["config_version"],
        "winner": None if valid_df.empty else valid_df.iloc[0]["model"],
        "winner_score": None if valid_df.empty else float(valid_df.iloc[0]["score"]),
        "backup": None if len(valid_df) < 2 else valid_df.iloc[1]["model"],
        "rejected": rejected_df.to_dict(orient="records"),
        "allowed_models": [] if valid_df.empty else valid_df["model"].tolist(),
    }
    (OUT / f"summary_{tag}.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))

    if not valid_df.empty:
        win = valid_df.iloc[0]["model"]
        win_pipe = clone(CANDIDATES[win]["pipeline"])
        win_pipe.fit(X_train_full, y_train_full)
        with open(OUT / f"pipeline_{tag}_main.pkl", "wb") as f:
            pickle.dump({"name": win, "pipeline": win_pipe}, f)
        if len(valid_df) >= 2:
            bk = valid_df.iloc[1]["model"]
            bk_pipe = clone(CANDIDATES[bk]["pipeline"])
            bk_pipe.fit(X_train_full, y_train_full)
            with open(OUT / f"pipeline_{tag}_backup.pkl", "wb") as f:
                pickle.dump({"name": bk, "pipeline": bk_pipe}, f)

    md = [f"# Сценарій: {scenario_cfg['scenario_name']}",
          f"Версія: `{scenario_cfg['config_version']}`", "",
          f"**Переможець:** `{summary['winner']}`  (score = {summary['winner_score']})",
          f"**Резерв:** `{summary['backup']}`", "",
          "## Допустимі моделі", valid_df.to_markdown(index=False), "",
          "## Відхилені моделі", rejected_df.to_markdown(index=False)]
    (OUT / f"report_{tag}.md").write_text("\n".join(md))
    return valid_df, rejected_df, summary

valid_fast, rej_fast, sum_fast = persist_scenario(SCENARIO_FAST, benchmark_df, "fast")
print("FAST winner:", sum_fast["winner"], "| backup:", sum_fast["backup"])
valid_fast

**Що зберігається автоматично:**

| Артефакт | Призначення |
|---|---|
| `ranking_<tag>.csv` | Рейтинг допустимих моделей (score, rank). |
| `rejected_<tag>.csv` | Список відхилених + причини. |
| `summary_<tag>.json` | Машинночитаний підсумок: winner, backup, config_version, rejected. |
| `report_<tag>.md` | Людиночитаний звіт. |
| `pipeline_<tag>_main.pkl` | Готова основна модель (fitted на повному train). |
| `pipeline_<tag>_backup.pkl` | Резервна модель (fitted на повному train). |

**Чому summary JSON — не просто «звіт», а інженерний контракт:**
- Містить `config_version` — якщо конфіг змінився, треба перерахувати рейтинг;
- Містить `winner` і `backup` — ці ідентифікатори підуть у metadata моделі в MLflow
  / S3 / реєстрі;
- Містить `rejected` — щоб через півроку ніхто не питав «чому ми не взяли SVC?».

**Backup — не «на всякий випадок», а вимога надійності:**
Якщо основна модель раптом не завантажилась або впала на edge-case, є готова заміна.
Вона вже навчена — перемикання займає секунди.

In [ ]:
valid_qual, rej_qual, sum_qual = persist_scenario(SCENARIO_QUALITY, benchmark_df, "quality")
print("QUALITY winner:", sum_qual["winner"], "| backup:", sum_qual["backup"])

compare = pd.DataFrame({
    "fast_allowed":    [valid_fast["model"].tolist()],
    "quality_allowed": [valid_qual["model"].tolist()],
    "fast_winner":     [sum_fast["winner"]],
    "quality_winner":  [sum_qual["winner"]],
    "fast_backup":     [sum_fast["backup"]],
    "quality_backup":  [sum_qual["backup"]],
}).T
print(compare)

**Що ми порівнюємо:**
1. **Множини допустимих моделей.** У `fast`-сценарії важкі моделі відсікаються
   жорсткішим лімітом latency. У `quality` — проходять, бо ліміт м'якший.
2. **Переможця.** У `fast` виграє компромісна модель (наприклад, `logreg`), бо latency
   важить 0.4. У `quality` виграє найточніша (`svc_rbf` або `rf`).
3. **Резерв.** Це друга за балом модель. Вона теж серіалізована й готова до роботи.
4. **Причини зміни рішення.** Якщо переможець той самий — значить, обмеження не було
   «вузьким місцем», або обидві моделі близькі за нормалізованим балом.

**Що означає «не змінився переможець»:**
- Або сценарії не розділяють кандидатів на різні боки;
- Або різниця в нормалізованих балах не перевищує різницю у вагах;
- Або найкраща модель виграє за обома критеріями (домінує).

**Практичний висновок:**
Один і той самий «benchmark_results.csv» може дати різних переможців залежно від SLA.
Тому **важливо** мати конфіг як окремий артефакт і версіонувати його.

In [ ]:
def sensitivity_analysis(benchmark_df, base_cfg, crit="roc_auc",
                         values=None, tag="fast"):
    if values is None:
        values = np.round(np.linspace(0.05, 0.90, 18), 3)
    rows = []
    for v in values:
        cfg = json.loads(json.dumps(base_cfg))
        others = [c for c in cfg["criteria"] if c != crit]
        old_others = sum(cfg["criteria"][c]["weight"] for c in others)
        cfg["criteria"][crit]["weight"] = float(v)
        if old_others > 0:
            k = (1.0 - v) / old_others
            for c in others:
                cfg["criteria"][c]["weight"] *= k
        s = sum(c["weight"] for c in cfg["criteria"].values())
        for c in cfg["criteria"].values():
            c["weight"] /= s

        valid, _ = rank_models(benchmark_df, cfg)
        if valid.empty:
            rows.append({"weight": v, "winner": None, "score": None}); continue
        rows.append({"weight": v,
                     "winner": valid.iloc[0]["model"],
                     "score": float(valid.iloc[0]["score"])})
    df = pd.DataFrame(rows)
    df.to_csv(OUT / f"sensitivity_{tag}_{crit}.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    for w in df["winner"].dropna().unique():
        sub = df[df["winner"] == w]
        ax.plot(sub["weight"], sub["score"], marker="o", label=w)
    ax.set_xlabel(f"Вага критерію {crit}"); ax.set_ylabel("Бал переможця")
    ax.set_title(f"Аналіз чутливості — сценарій {tag}")
    ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.savefig(OUT / f"sensitivity_{tag}_{crit}.png", dpi=120); plt.show()

    wins = df["winner"].value_counts().to_dict()
    switches = df[df["winner"] != df["winner"].shift()][["weight", "winner"]].dropna()
    return df, wins, switches

sens_df, wins, switches = sensitivity_analysis(benchmark_df, SCENARIO_FAST, "roc_auc", tag="fast")
print("Кількість перемог:", wins)
print("Точки зміни переможця:\n", switches)

**Що ми робимо:**
- Беремо одну «суттєву» вагу (тут — `roc_auc` у `fast`-сценарії).
- Пробігаємо діапазон 0.05 → 0.90.
- Решту ваг автоматично перераховуємо так, щоб сума = 1.0
  (пропорційно зменшуємо залишкові).

**Що показує графік:**
- **Плато** — модель виграє стабільно, рішення стійке.
- **Стрибок** — точка, у якій перше місце переходить до іншої моделі.
- **Перетин** — «межа чутливості»: якщо вага коливається в межах ±0.05, рішення
  не зміниться, і ним можна користуватись.

**Як інтерпретувати `wins`:**
- Якщо одна модель виграє у 15 з 18 точок — рішення стійке;
- Якщо приблизно 50/50 — вага надто впливова, треба перевірити, чи справді
  вона така важлива, чи це артефакт нормалізації.

**Точки зміни (`switches`):**
Явно показують, при якій вазі ламається поточний вибір. Це критично для
комунікації зі стейкхолдерами: «Якщо ви вважаєте latency важливішою за 0.6 —
модель зміниться на іншу».

In [ ]:
import unittest

class TestRankingLogic(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.cfg = {
            "scenario_name": "t", "config_version": "t",
            "constraints": {"accuracy_min": 0.80, "p95_predict_ms_max": 50.0},
            "criteria": {
                "roc_auc":        {"direction": "maximize", "weight": 0.5, "min_ref": 0.5, "max_ref": 1.0},
                "p95_predict_ms": {"direction": "minimize", "weight": 0.5, "min_ref": 0.0, "max_ref": 100.0},
            },
        }

    def _df(self, rows): return pd.DataFrame(rows)

    def test_maximize_norm(self):
        self.assertAlmostEqual(normalize(0.75, "maximize", 0.5, 1.0), 0.5)

    def test_minimize_norm(self):
        self.assertAlmostEqual(normalize(50.0, "minimize", 0.0, 100.0), 0.5)

    def test_clip_out_of_range(self):
        self.assertEqual(normalize(2.0, "maximize", 0.5, 1.0), 1.0)
        self.assertEqual(normalize(-1.0, "maximize", 0.5, 1.0), 0.0)
        self.assertEqual(normalize(999.0, "minimize", 0.0, 100.0), 0.0)

    def test_boundary_allowed(self):
        ok, v = check_constraints({"accuracy": 0.80, "p95_predict_ms": 50.0},
                                  self.cfg["constraints"])
        self.assertTrue(ok, v)

    def test_reject_bad_model(self):
        df = self._df([
            {"model": "good", "roc_auc": 0.9, "p95_predict_ms": 10.0, "accuracy": 0.9},
            {"model": "bad",  "roc_auc": 0.9, "p95_predict_ms": 10.0, "accuracy": 0.5},
        ])
        valid, rej = rank_models(df, self.cfg)
        self.assertIn("bad", rej["model"].tolist())
        self.assertNotIn("bad", valid["model"].tolist())

    def test_bad_weight_sum(self):
        bad = json.loads(json.dumps(self.cfg))
        bad["criteria"]["roc_auc"]["weight"] = 0.9
        with self.assertRaises(AssertionError):
            validate_config(bad)

    def test_missing_metric(self):
        df = self._df([{"model": "x", "accuracy": 0.9, "p95_predict_ms": 10.0}])
        _, rej = rank_models(df, self.cfg)
        self.assertTrue(any("Відсутні метрики" in v for v in rej["violations"]))

    def test_no_valid_models(self):
        df = self._df([{"model": "x", "roc_auc": 0.1, "p95_predict_ms": 999.0, "accuracy": 0.1}])
        valid, rej = rank_models(df, self.cfg)
        self.assertTrue(valid.empty)
        self.assertEqual(len(rej), 1)

    def test_ties_deterministic(self):
        df = self._df([
            {"model": "b", "roc_auc": 0.8, "p95_predict_ms": 20.0, "accuracy": 0.9},
            {"model": "a", "roc_auc": 0.8, "p95_predict_ms": 20.0, "accuracy": 0.9},
        ])
        v, _ = rank_models(df, self.cfg)
        self.assertEqual(v.iloc[0]["model"], "a")

    def test_load_saved_pipeline(self):
        path = OUT / "pipeline_fast_main.pkl"
        self.assertTrue(path.exists(), "pipeline не збережено")
        obj = pickle.loads(path.read_bytes())
        preds = obj["pipeline"].predict(X_test.iloc[:10])
        self.assertEqual(len(preds), 10)

suite = unittest.TestLoader().loadTestsFromTestCase(TestRankingLogic)
res = unittest.TextTestRunner(verbosity=1).run(suite)
print("Успішно:", res.wasSuccessful(), "| провалів:", len(res.failures))

**10 тестів — по одному на кожен «біль» інженера:**

| # | Тест | Що захищає |
|---|---|---|
| 1 | `maximize_norm` | Формула `(v - min) / (max - min)`. |
| 2 | `minimize_norm` | Дзеркальна формула для «менше краще». |
| 3 | `clip_out_of_range` | Значення за межами не ламають шкалу. |
| 4 | `boundary_allowed` | `=` на межі → допустимо (SLA «≤»). |
| 5 | `reject_bad_model` | Модель з низькою метрикою не потрапляє в ranking. |
| 6 | `bad_weight_sum` | Конфіг з неправильною сумою ваг відкидається. |
| 7 | `missing_metric` | Відсутність метрики → у rejected, а не в ranking. |
| 8 | `no_valid_models` | Порожній ranking не ламає код. |
| 9 | `ties_deterministic` | Однакові бали → сортування за алфавітом. |
| 10 | `load_saved_pipeline` | Серіалізований pipeline реально працює. |

**Принципи:**
- Тести **не тренують ML-моделі**. Логіка вибору перевіряється на крихітних таблицях.
- Тільки тест 10 торкається реального pipeline — і то лише `predict` на 10 зразках.
- Якщо тест впав — спершу читайте `res.failures[0][1]` (traceback), там буде видно,
  яке саме значення не зійшлось.

In [ ]:
cv_auc = float(cv_results[sum_fast["winner"]]["agg"]["roc_auc_mean"])
cv_auc_std = float(cv_results[sum_fast["winner"]]["agg"]["roc_auc_std"])
expected_auc = (cv_auc - 2 * cv_auc_std, cv_auc + 2 * cv_auc_std)
print(f"Очікуваний ROC-AUC на test ≈ {cv_auc:.3f} ± {2*cv_auc_std:.3f}")
print("Ризики: розбіжність train/test розподілу; HW-залежна latency; можливий overfit k-NN.")

with open(OUT / "pipeline_fast_main.pkl", "rb") as f:
    main_obj = pickle.load(f)
best_pipe = main_obj["pipeline"]
best_name = main_obj["name"]

y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]
test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
print(f"\n[{best_name}] TEST:")
for k, v in test_metrics.items():
    print(f"  {k:9s} = {v:.4f}")

in_range = expected_auc[0] <= test_metrics["roc_auc"] <= expected_auc[1]
print(f"\nROC-AUC у межах CV-очікування: {in_range}")
print("Жодних змін ваг/меж/кандидатів після перегляду не вносилось.")

**Правила (порушувати не можна):**
1. **Спочатку фіксуємо очікування** — `expected_auc = mean ± 2·std` з CV.
2. **Потім** дивимось test. Порядок критичний: інакше почнеться «підгонка під test».
3. Після перегляду — **ніяких змін** у вагах, межах, кандидатах. Якщо хочеться
   щось змінити — це **нова версія** експерименту з іншим `config_version`.

**Як читати результат:**
- `ROC-AUC у межах CV-очікування: True` → все добре, модель узагальнюється.
- Якщо `test_auc < cv_auc - 2·std` → є розбіжність train/test, треба:
  - перевірити дрейф ознак;
  - перевірити, чи test не має іншого розподілу класів;
  - розглянути регуляризацію або більш просту модель.
- Якщо `test_auc > cv_auc + 2·std` → можливо, CV-фолди занадто «важкі», або
  пощастило з конкретним test-сплітом (рідко, але буває).

**Інженерний висновок:** результат test — це «касовий чек» для CV-очікувань, а не
привід щось переналаштовувати.

## Розділ 16 — Підсумковий інженерний висновок

### 1. Хто порушив жорсткі обмеження
Див. `artifacts/rejected_fast.csv` та `artifacts/rejected_quality.csv`.
Зазвичай:
- `dummy` — не проходить `roc_auc_min` (немає сигналу);
- `svc_rbf` / `rf` — можуть провалити `p95_predict_ms_max` у `fast`-сценарії;
- `rf` — може перевищити `artifact_size_mb_max`, якщо дерева глибокі.

### 2. Основна і резервна моделі
Зафіксовано у `summary_fast.json` та `summary_quality.json` (поля `winner`, `backup`).
Обидва pipeline серіалізовано окремо для швидкого перемикання.

### 3. Які критерії визначили результат
- `fast_inference`: ROC-AUC (0.4) + p95-latency (0.4) + size (0.2);
- `quality_first`: ROC-AUC (0.6) + accuracy (0.3) + p95-latency (0.1).

### 4. Чи змінився переможець у другому сценарії
Так, якщо точніша, але повільніша модель раніше відсікалась/програвала за latency.
Якщо ні — обмеження не було активним, і обидві моделі домінують одна одну
по різних критеріях.

### 5. Чутливість до ваг
Графік `sensitivity_fast_roc_auc.png` + таблиця `sensitivity_fast_roc_auc.csv`.
- Якщо переможець стабільний у діапазоні 0.2–0.6 — рішення стійке;
- Якщо «стрибок» уже при 0.3 — вага надто впливова, треба уважніше з нею.

### 6. Чи підтвердив test set CV-очікування
`True/False` з клітинки 14. True → модель узагальнюється; False → розбіжність
розподілів або переоцінка якості на CV.

### 7. Які показники залежать від апаратного середовища
- **Залежать:** `train_time_s`, `p50_predict_ms`, `p95_predict_ms`, `throughput_sps`;
- **Не залежать:** `accuracy`, `f1`, `roc_auc`, `artifact_size_mb`.

Тому для порівняння кандидатів у CI варто фіксувати середовище (Docker + pinned CPU).

### 8. Коли перезапускати процедуру
- Дрейф ознак (`PSI > 0.2`);
- Падіння ROC-AUC на моніторингу > 5 п.п.;
- Зміна SLA (latency/size);
- Плановий ретрейн за cadence (> 1 місяця);
- Зміна схеми даних або поява нового класу.

### 9. Обмеження розробленого інструменту
- Один test-погляд на версію конфігу; після — тільки новий `config_version`;
- Latency вимірюється на **одному** фіксованому батчі (200) — не враховує
  конкурентне навантаження і cold-start;
- Пам'ять (RSS, peak) **не вимірюється** — фіксується лише розмір артефакту;
- Пропускна здатність залежить від CPU; при переносі на інший клас машин
  числа треба перезаміряти;
- Детермінізм зберігається завдяки `random_state`, але тільки в межах
  однакових версій бібліотек.

# Розділ 16 — Підсумковий інженерний висновок

> **Головний факт:** обидва сценарії (`fast_inference` і `quality_first`) завершились
> **без переможця**. `valid_df` порожній, `winner=None`, `backup=None`. Це **валідний
> результат** процедури: жоден кандидат не проходить жорсткі обмеження. Далі — розбір
> на 9 питань.

---

## 1. Хто порушив жорсткі обмеження

### Сценарій `fast_inference`
Обмеження: `accuracy ≥ 0.80`, `roc_auc ≥ 0.85`, `p95 ≤ 50 мс`, `train ≤ 60 с`, `size ≤ 100 МБ`.

| Модель | accuracy | roc_auc | p95, мс | Що провалено |
|---|---|---|---|---|
| `dummy`   | 0.5046 | 0.5000 | 0.56   | ❌ `accuracy < 0.80`, ❌ `roc_auc < 0.85` |
| `logreg`  | 0.6812 | **0.9585** | 4.79 | ❌ `accuracy < 0.80` (єдине порушення) |
| `knn`     | 0.5583 | 0.5662 | 8.53   | ❌ `accuracy`, ❌ `roc_auc` |
| `svc_rbf` | 0.5140 | 0.4795 | 42.95  | ❌ `accuracy`, ❌ `roc_auc` |
| `rf`      | 0.6079 | 0.6339 | 124.40 | ❌ `accuracy`, ❌ `roc_auc`, ❌ `p95 > 50` |

**Усі 5 кандидатів відхилені.** Вирішальне обмеження — `accuracy_min = 0.80`: жодна
модель його не досягає (максимум 0.6812 у `logreg`). Сценарій **нерозв'язний за побудовою**.

### Сценарій `quality_first`
Обмеження: `accuracy ≥ 0.85`, `roc_auc ≥ 0.90`, `p95 ≤ 200 мс`, `size ≤ 200 МБ`.

| Модель | accuracy | roc_auc | Що провалено |
|---|---|---|---|
| `dummy`   | 0.5046 | 0.5000 | ❌ обидва |
| `logreg`  | 0.6812 | **0.9585** | ❌ `accuracy < 0.85` (ROC-AUC ✅) |
| `knn`     | 0.5583 | 0.5662 | ❌ обидва |
| `svc_rbf` | 0.5140 | 0.4795 | ❌ обидва |
| `rf`      | 0.6079 | 0.6339 | ❌ обидва |

**Знову порожньо.** `logreg` — єдина модель, що майже проходить (усе ок, крім `accuracy`).

---

## 2. Яку основну і резервну моделі вибрано

**Не вибрано жодної.** Обидва `summary_*.json` мають `winner: null`, `backup: null`,
`allowed_models: []`. `pipeline_fast_main.pkl` і `pipeline_fast_backup.pkl` **не створені** —
тому тест `test_load_saved_pipeline` впав з `AssertionError: pipeline не збережено`.

**Інженерна знахідка:** процедура **правильно** відмовилась обирати модель, коли жодна
не відповідає SLA. Альтернатива (вибрати «найменш поганого») приховала б реальну
проблему — нереалістичні обмеження.

**Що було б вибрано при реалістичних порогах:**
- `fast_inference`: **`logreg`** (winner) + `knn` або `rf` (backup);
- `quality_first`: **`logreg`** (winner) + `rf` (backup) — `logreg` має найвищий ROC-AUC.

---

## 3. Які критерії визначили результат

**Порядок впливу обмежень:**

1. **`accuracy_min = 0.80` / `0.85`** — абсолютно домінуюче обмеження. Саме воно відкидає
   `logreg` — єдиного реального претендента. Решта моделей відкидається цим же + `roc_auc_min`.
2. **`roc_auc_min = 0.85` (FAST) / `0.90` (QUALITY)** — друге за силою. Відсіює `dummy`,
   `knn`, `svc_rbf`, `rf` у FAST. У QUALITY `logreg` його проходить (0.9585 ≥ 0.90).
3. **`p95_predict_ms_max = 50` (FAST)** — відсіює `rf` (124 мс). `svc_rbf` (42.95 мс)
   проходить, але добивається `accuracy`.
4. **`train_time_s_max`, `artifact_size_mb_max`** — не спрацювали жодного разу.
   Усі моделі навчаються за < 1.2 с і важать < 8 МБ.

---

## 4. Чи змінився переможець у другому сценарії

**Ні, бо переможця немає ні в першому, ні в другому.** Таблиця `compare` повністю порожня:
`fast_allowed=[]`, `quality_allowed=[]`, обидва winner'и `None`.

**Чому:** обидва сценарії вперлися в **те саме вузьке місце** — низька точність усіх
моделей. Різниця між сценаріями не проявилась, бо процедура зупинилась раніше, ніж
дійшла до порівняння.

---

## 5. Наскільки рішення чутливе до ваг

**Аналіз чутливості не відбувся** — графік порожній, `Кількість перемог: {}`,
`switches: Empty DataFrame`.

**Причина:** `sensitivity_analysis` викликає `rank_models`, а той повертає порожній
`valid_df` — отже, `winner = None` для всіх 18 значень ваги.

**Інженерний висновок:** обмеження **повністю перекривають** вплив ваг. Поки
`accuracy_min = 0.80` відсіює `logreg`, зміна ваги `roc_auc` нічого не змінить.
**Правильний порядок:** спочатку реалістичні constraints, потім — аналіз чутливості.

---

## 6. Чи підтвердив test set CV-очікування

**Ні, і не міг підтвердити.** Клітинка фінальної перевірки впала з `KeyError: None`,
бо `sum_fast["winner"] is None` → `cv_results[None]` → KeyError. Далі ланцюжок:
`pipeline_fast_main.pkl` не існує → `test_load_saved_pipeline` фейлиться → test set
**взагалі не використаний**.

**Позитивний момент:** жодних маніпуляцій з test set не сталося, тому він залишається
«чистим» для нової версії експерименту.

---

## 7. Які показники залежать від апаратного середовища

| Метрика | Залежить від HW | Обґрунтування |
|---|---|---|
| `train_time_s`     | ✅ Так | Час CPU/GPU, паралелізм `n_jobs=-1` |
| `p50_predict_ms`   | ✅ Так | Тактова частота, кеш, JIT |
| `p95_predict_ms`   | ✅ Так | Те саме + OS-шум, GC, інші процеси |
| `throughput_sps`   | ✅ Так | Обернена до latency |
| `accuracy`, `f1`, `roc_auc` | ❌ Ні | Залежать лише від даних і алгоритму |
| `artifact_size_mb` | ❌ Ні | Pickle детермінований |

**Спостереження:** у різних запусках той самий `rf` дав `p95 = 113 мс` і `124 мс` —
**~10% різниці лише від середовища**. Тому latency треба порівнювати у **фіксованому
середовищі** (Docker + pinned CPU).

---

## 8. Коли процедуру потрібно запустити повторно

**Терміново (з виправленням конфігурації):**
1. **Скоригувати `accuracy_min`** з 0.80/0.85 до реалістичного рівня (наприклад,
   **0.65** для FAST, **0.70** для QUALITY).
2. **Розслідувати аномалію `ROC-AUC = 0.96` vs `accuracy = 0.68`** у `logreg`:
   - поріг 0.5 не оптимальний (треба тюнити threshold);
   - класи мають дисбаланс після об'єднання train+test;
   - `y` містить прихований третій стан.

**Стандартні тригери (з розділу 1):**
- Дрейф ознак `PSI > 0.2`;
- Падіння `ROC-AUC` у продакшені > 5 п.п.;
- Зміна SLA (latency/size);
- Плановий ретрейн за cadence (> 1 місяця);
- Зміна схеми даних або поява нового класу.

**При кожному рестарті:** обов'язково `config_version += 1`.

---

## 9. Які обмеження має розроблений інструмент

**Методологічні:**
1. **Один test-погляд на версію конфігу** — якщо спалено, потрібна нова версія.
2. **Latency вимірюється на одному фіксованому батчі (200)** — не враховує cold-start,
   конкурентне навантаження, різні розміри пакетів.
3. **Пам'ять (RSS/peak) не вимірюється** — лише розмір pickle.
4. **Час залежить від HW** — порівнювати latency між машинами некоректно.
5. **Детермінізм — лише в межах однакових версій бібліотек.**

**Специфічні, виявлені на цьому запуску:**
6. **Constraints не перевіряються на реалістичність ДО запуску.** Інструмент не
   попереджає, що `accuracy_min = 0.80` недосяжний. Варто додати перевірку: якщо
   **усі** моделі порушують **одне й те саме** обмеження — це привід переглянути його.
7. **Аналіз чутливості не має fallback** — при порожньому `valid_df` має явно
   повідомляти «ranking порожній, спочатку послабте constraints», а не малювати
   порожній графік.
8. **Gap `accuracy` vs `ROC-AUC` не детектується.** Варто додати warning:
   `|AUC − 0.5| ≫ |acc − base_rate|` → підозра на проблему з порогом або мітками.
9. **Об'єднання `Training + Testing`** у клітинці 1 потребує документування:
   якщо оригінальний test мав інший розподіл, об'єднання його «затірає».
10. **Немає автоматичної перевірки схеми даних.** Варто додати
    `assert set(y.unique()) == {0, 1}` та `assert not X.isna().any().any()`.

---

## Підсумок одним абзацом

Процедура **спрацювала коректно** — вона виявила, що за жорстких вимог
`accuracy ≥ 0.80` (FAST) і `≥ 0.85` (QUALITY) жодна з п'яти моделей не проходить.
`logreg` — єдина модель із реально високим ROC-AUC (0.9585) і низькою latency (4.79 мс),
але її `accuracy` (0.6812) провалює пороги. Обидва сценарії залишились без переможця,
`pipeline_*.pkl` не створені, test set не торкнуто. **Наступний крок** — переглянути
`accuracy_min` у бік реалістичності (0.65 / 0.70) і розслідувати аномалію
«високий AUC / низька accuracy». Після цього процедура дасть переможця
(`logreg` у FAST, `logreg` або `rf` у QUALITY), а аналіз чутливості стане інформативним.